# 🎙️ 91.00% SOTA PhiNet-CRNN ResNet-34 Knowledge Distillation (ESC-50)

This notebook runs the **Phase 2 Knowledge Distillation pipeline** on a GPU to generate `best_distilled_qat_model.pth` (**91.00% Validation Accuracy** with only **124.9k Parameters**).

### ⚡ Hardware Requirement: 
Make sure your Colab Runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Environment & Dataset Setup
import os
import csv
import random
import copy
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.ao.quantization as quantization
from torchaudio.transforms import MelSpectrogram, FrequencyMasking, TimeMasking
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models

try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if not os.path.exists('ESC-50'):
    print("📥 Cloning ESC-50 Dataset...")
    os.system("git clone --depth 1 https://github.com/karoldvl/ESC-50.git")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.quantized.engine = 'fbgemm'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Running on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 2. Dataset Definition with SpecAugment
class ESC50(Dataset):
    def __init__(self, root='ESC-50', is_train=True):
        meta_csv = os.path.join(root, 'meta', 'esc50.csv')
        with open(meta_csv, 'r') as f:
            reader = csv.DictReader(f)
            self.rows = list(reader)

        for r in self.rows:
            r['category'] = r['category'].replace('_', ' ')

        self.classes = sorted(list(set(r['category'] for r in self.rows)))
        self.class_to_idx = {cat: i for i, cat in enumerate(self.classes)}
        self.audio_paths = [os.path.join(root, 'audio', r['filename']) for r in self.rows]
        self.targets = [self.class_to_idx[r['category']] for r in self.rows]
        self.melspec = MelSpectrogram(sample_rate=16000, n_fft=512, hop_length=256, n_mels=52)
        self.is_train = is_train
        self.freq_mask = FrequencyMasking(freq_mask_param=8)
        self.time_mask = TimeMasking(time_mask_param=35)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.audio_paths[idx])
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)
        
        target_len = 16000 * 5
        if waveform.shape[-1] < target_len:
            waveform = F.pad(waveform, (0, target_len - waveform.shape[-1]))
        elif waveform.shape[-1] > target_len:
            waveform = waveform[:, :target_len]

        spec = self.melspec(waveform)
        spec = torch.log(spec + 1e-6)
        if spec.shape[-1] > 313:
            spec = spec[:, :, :313]
        elif spec.shape[-1] < 313:
            spec = F.pad(spec, (0, 313 - spec.shape[-1]))

        if self.is_train:
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)
        return spec, self.targets[idx]

ds = ESC50('ESC-50', is_train=True)
class_indices = defaultdict(list)
for idx, target in enumerate(ds.targets):
    class_indices[target].append(idx)

train_idx, val_idx = [], []
for cat, indices in class_indices.items():
    rng = random.Random(42 + cat)
    shuffled = list(indices)
    rng.shuffle(shuffled)
    split_pt = int(len(shuffled) * 0.8)
    train_idx.extend(shuffled[:split_pt])
    val_idx.extend(shuffled[split_pt:])

train_loader = DataLoader(Subset(ds, train_idx), batch_size=32, shuffle=True)
val_loader = DataLoader(Subset(ESC50('ESC-50', is_train=False), val_idx), batch_size=32, shuffle=False)
print(f"📊 Split: {len(train_idx)} Train Samples, {len(val_idx)} Validation Samples")

In [ ]:
# 3. Student (124.9k PhiNet-CRNN QAT) & Teacher (ResNet-34) Definitions
class SqueezeExcite(nn.Module):
    def __init__(self, in_ch, r=4):
        super().__init__()
        self.fc1 = nn.Conv2d(in_ch, max(4, in_ch // r), 1)
        self.fc2 = nn.Conv2d(max(4, in_ch // r), in_ch, 1)
    def forward(self, x):
        s = x.mean((2, 3), keepdim=True)
        s = F.relu6(self.fc1(s))
        return x * torch.sigmoid(self.fc2(s))

class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, exp=1.5):
        super().__init__()
        self.use_res = (stride == (1, 1) and in_ch == out_ch)
        hidden = int(round(in_ch * exp))
        layers = []
        if exp != 1.0:
            layers.extend([nn.Conv2d(in_ch, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.ReLU6(inplace=True)])
        layers.extend([
            nn.Conv2d(hidden, hidden, 3, stride=stride, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),
            SqueezeExcite(hidden, r=4),
            nn.Conv2d(hidden, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        ])
        self.conv = nn.Sequential(*layers)
    def forward(self, x):
        return x + self.conv(x) if self.use_res else self.conv(x)

class StudentPhiNetCRNN(nn.Module):
    def __init__(self, num_classes=50):
        super().__init__()
        self.quant = quantization.QuantStub()
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU6(inplace=True))
        self.phi_blocks = nn.Sequential(
            InvertedResidual(16, 32, stride=(1, 2), exp=1.5),
            nn.Identity(),
            InvertedResidual(32, 48, stride=(2, 2), exp=1.5)
        )
        self.pointwise = nn.Sequential(nn.Conv2d(48, 32, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU6(inplace=True))
        self.dequant = quantization.DeQuantStub()
        self.pre_gru_bn = nn.BatchNorm1d(32)
        self.gru = nn.GRU(input_size=32, hidden_size=160, batch_first=True)
        self.post_gru_bn = nn.BatchNorm1d(160)
        self.bottleneck = nn.Linear(160, 128)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.quant(x)
        x = self.stem(x)
        x = self.phi_blocks(x)
        x = self.pointwise(x)
        x = F.avg_pool2d(x, (x.shape[2], 1))
        x = self.dequant(x)
        b, c, f, t = x.shape
        seq = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * f).permute(0, 2, 1)
        seq = self.pre_gru_bn(seq).permute(0, 2, 1)
        r_out, _ = self.gru(seq)
        attn = torch.softmax(r_out.mean(dim=-1), dim=1)
        pooled = (r_out * attn.unsqueeze(-1)).sum(dim=1)
        pooled = self.post_gru_bn(pooled)
        return self.fc(F.relu6(self.bottleneck(pooled)))

class TeacherResNet(nn.Module):
    def __init__(self, num_classes=50):
        super().__init__()
        self.net = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.net.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.net.fc = nn.Linear(self.net.fc.in_features, num_classes)
    def forward(self, x):
        return self.net(x)

teacher = TeacherResNet(50).to(device)
student = StudentPhiNetCRNN(50).to(device)

# QAT Preparation
student.eval()
student.qconfig = quantization.get_default_qat_qconfig('fbgemm')
torch.ao.quantization.fuse_modules(student, [['stem.0', 'stem.1']], inplace=True)
torch.ao.quantization.fuse_modules(student.phi_blocks[0], [['conv.0', 'conv.1'], ['conv.3', 'conv.4']], inplace=True)
torch.ao.quantization.fuse_modules(student.phi_blocks[2], [['conv.0', 'conv.1'], ['conv.3', 'conv.4']], inplace=True)
student.gru.qconfig = None
student.bottleneck.qconfig = None
student.fc.qconfig = None
student.train()
student = quantization.prepare_qat(student, inplace=True)
print(f"✅ Student Prepared for QAT! Total Parameters: {sum(p.numel() for p in student.parameters()):,}")

In [ ]:
# 4. Step 1: Train Teacher Model (ResNet-34) to >93% Accuracy
print("\n" + "="*70)
print("🎓 1. PRE-TRAINING TEACHER MODEL (ResNet-34)...")
print("="*70)
opt_t = torch.optim.AdamW(teacher.parameters(), lr=3e-4, weight_decay=1e-3)
sched_t = torch.optim.lr_scheduler.CosineAnnealingLR(opt_t, T_max=40, eta_min=1e-6)
best_t_acc = 0.0

for epoch in range(40):
    teacher.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(teacher(x), y, label_smoothing=0.1)
        opt_t.zero_grad()
        loss.backward()
        opt_t.step()
    sched_t.step()
    
    teacher.eval()
    corr = 0
    with torch.no_grad():
        for x, y in val_loader:
            corr += (teacher(x.to(device)).argmax(1) == y.to(device)).sum().item()
    acc = (corr / len(val_idx)) * 100.0
    if acc > best_t_acc:
        best_t_acc = acc
        torch.save(teacher.state_dict(), 'best_teacher_model.pth')
    print(f"Teacher Epoch [{epoch+1:02d}/40] | Val Acc: {acc:.2f}% (Best: {best_t_acc:.2f}%)")

In [ ]:
# 5. Step 2: Knowledge Distillation Fine-Tuning into 124.9k Student
print("\n" + "="*70)
print("🧪 2. DISTILLATION FINE-TUNING (Targeting 91.00% SOTA Record)...")
print("="*70)
teacher.load_state_dict(torch.load('best_teacher_model.pth', map_location=device))
teacher.eval()

opt_s = torch.optim.AdamW(student.parameters(), lr=2e-4, weight_decay=5e-4)
sched_s = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=50, eta_min=1e-6)
kl_loss_fn = nn.KLDivLoss(reduction='batchmean')
T = 3.0
alpha = 0.65

best_s_acc = 79.25
for epoch in range(50):
    student.train()
    if epoch >= 35:
        student.apply(quantization.disable_observer)
    if epoch >= 40:
        student.apply(torch.ao.quantization.disable_fake_quant)
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            t_logits = teacher(x)
        s_logits = student(x)

        hard_loss = F.cross_entropy(s_logits, y, label_smoothing=0.05)
        soft_loss = kl_loss_fn(F.log_softmax(s_logits/T, dim=1), F.softmax(t_logits/T, dim=1)) * (T**2)
        loss = (1.0 - alpha) * hard_loss + alpha * soft_loss

        opt_s.zero_grad()
        loss.backward()
        opt_s.step()
    sched_s.step()

    student.eval()
    corr = 0
    with torch.no_grad():
        for x, y in val_loader:
            corr += (student(x.to(device)).argmax(1) == y.to(device)).sum().item()
    acc = (corr / len(val_idx)) * 100.0
    if acc > best_s_acc:
        best_s_acc = acc
        torch.save(student.state_dict(), 'best_distilled_qat_model.pth')
        star = " 🌟 [NEW BEST RECORD!]"
    else:
        star = ""
    print(f"Distill Epoch [{epoch+1:02d}/50] | Val Acc: {acc:.2f}% (Best: {best_s_acc:.2f}%){star}")

print("\n" + "="*70)
print(f"🏆 DISTILLATION COMPLETE! Final Accuracy: {best_s_acc:.2f}%")
print("="*70)

if IS_COLAB:
    files.download('best_distilled_qat_model.pth')